In [1]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/qwenInstData.xlsx" .

In [2]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [3]:
df = pd.read_excel("qwenInstData.xlsx")

In [4]:
ref_col = 'Hop_3_Tactic (Ref)'
pred_col = 'Hop_3_Tactic'

In [5]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
print(len(df))

846


In [6]:
refTactics = []
for refTactic in df[ref_col]:
  refTactics.extend(ast.literal_eval(refTactic))
set(refTactics)

{'انسان\u200cزدایی',
 'تحریک به خشونت/آسیب',
 'ترویج کلیشه\u200cهای مضر',
 'دشنام/زبان تحقیرآمیز',
 'فراخوان برای تبعیض/حذف',
 'نظریه\u200cهای توطئه'}

In [7]:
llmTactics = []
for llmTactic in df[pred_col]:
  llmTactics.extend(ast.literal_eval(llmTactic))
set(llmTactics)

{'آسیب\u200cزدایی',
 'انحطاط',
 'انسان\u200cزدایی',
 'تحریک به خشونت/آسیب',
 'تخریب شخصیت',
 'تخریب شخصیتی',
 'تخریب مصداقیت',
 'تخریب هویت',
 'تخریب هویت اجتماعی',
 'تخریب هویت فردی',
 'تخریب هویت مذهبی',
 'تخریب هویت گروهی',
 'ترجیح به نام\u200cهای توهین\u200cآمیز',
 'ترجیح دادن گروه خود به\u200cجای دیگران',
 'ترولینگ/تهدید ضمنی',
 'ترویج کلیشه\u200cهای مضر',
 'تقلیل به سطح مادی',
 'تهدید به آسیب',
 'تهدید به خشونت',
 'تهدید به خشونت/آسیب',
 'تهدید/تحریک به خشونت',
 'توصیف به عنوان تهدید',
 'خیانت',
 'دشنام/زبان تحقیرآمیز',
 'فراخوان برای تبعیض/حذف',
 'نظریه\u200cهای توطئه',
 'نفرت\u200cپراکنی مبتنی بر هویت خانوادگی',
 'نقد به\u200cصورت ذاتی و غیرقابل تغییر',
 'نقد به\u200cصورت شخصی و اتهام\u200cزنی',
 'نقد به\u200cصورت شخصی و توهین به هویت',
 'نقد به\u200cصورت کلی\u200cگرایانه',
 'نقد سازمانی بی\u200cپایه',
 'نقد سیاسی به\u200cصورت توهین',
 'نقد سیاسی به\u200cعنوان توهین هویتی',
 'نقد سیاسی بی\u200cاحترامی',
 'نقد سیاسی بی\u200cپایه',
 'نقد سیاسی شدید',
 'نقد سیاسی مبتنی بر توهین',


In [8]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [9]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df[ref_col]) + list(df[pred_col]))

y_true = mlb.transform(df[ref_col])
y_pred = mlb.transform(df[pred_col])

In [10]:
def f1_adapted(y_true_bin, y_pred_bin):
    """
    Adapted F1 from arXiv:2506.19073 using binarized arrays.
    A prediction is correct if ANY label overlaps for a sample.
    """

    hits = []

    for yt, yp in zip(y_true_bin, y_pred_bin):
        # skip empty ground truth (recommended)
        if not yt.any():
            continue

        hit = int((yt & yp).any())
        hits.append(hit)

    return np.mean(hits) if hits else 0.0

In [11]:
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_adapted = f1_adapted(y_true, y_pred)

print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")
print(f"Adapted F1: {f1_adapted:.4f}")

Micro F1: 0.6157
Macro F1: 0.0702
Adapted F1: 0.9421
